In [1]:
#============================================
# Celda 1 — Carga parquets raw
#============================================
import pandas as pd, numpy as np, os
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print(f"✅ ROOT: {ROOT}")

PATH_AEMET = Path('output/ambiental/01-raw/raw_aemet_estaciones.parquet')
PATH_WAQI  = Path('output/ambiental/raw_waqi_ciudades.parquet')

df_waqi  = pd.read_parquet(PATH_WAQI)
df_aemet = pd.read_parquet(PATH_AEMET) if PATH_AEMET.exists() else pd.DataFrame()

AEMET_OK = not df_aemet.empty
WAQI_OK  = not df_waqi.empty

print(f"{'✅' if AEMET_OK else '⚠️ '} AEMET : {len(df_aemet):>6} filas × {len(df_aemet.columns)} cols")
print(f"{'✅' if WAQI_OK  else '❌ '} WAQI  : {len(df_waqi):>6}  filas × {len(df_waqi.columns)} cols")

✅ ROOT: /workspaces/TFG_Spain-s_Migratory_Flow
⚠️  AEMET :      0 filas × 0 cols
✅ WAQI  :     30  filas × 18 cols


In [2]:
#============================================
# Celda 2 — Limpieza df_waqi
#============================================

# Tipos
df_waqi['timestamp'] = pd.to_datetime(df_waqi['timestamp'], errors='coerce')

cols_num = ['lat','lon','aqi','NO2','PM25','PM10','O3','SO2','CO',
            'temperatura','humedad','presion','viento']
for c in cols_num:
    if c in df_waqi.columns:
        df_waqi[c] = pd.to_numeric(df_waqi[c], errors='coerce')

# Deduplicar
antes = len(df_waqi)
df_waqi = df_waqi.drop_duplicates(subset=['ciudad', 'timestamp'])
print(f"   Duplicados eliminados: {antes - len(df_waqi)}")

# Columnas temporales útiles
df_waqi['fecha'] = df_waqi['timestamp'].dt.date
df_waqi['hora']  = df_waqi['timestamp'].dt.hour

# Estandarizar nivel_aqi si no existe
if 'nivel_aqi' not in df_waqi.columns:
    bins   = [0, 50, 100, 150, 200, 300, 500]
    labels = ['Bueno','Moderado','Insalubre_sensibles','Insalubre','Muy_insalubre','Peligroso']
    df_waqi['nivel_aqi'] = pd.cut(df_waqi['aqi'], bins=bins, labels=labels)

print(f"✅ df_waqi limpio: {df_waqi.shape}")
print(f"   Nulos: {round(df_waqi.isnull().sum().sum()/df_waqi.size*100,1)}%")
print(f"   Rango timestamp: {df_waqi['timestamp'].min()} → {df_waqi['timestamp'].max()}")
df_waqi.head()

   Duplicados eliminados: 0
✅ df_waqi limpio: (30, 20)
   Nulos: 3.5%
   Rango timestamp: 2026-06-11 11:00:00 → 2026-06-20 12:00:00


,ciudad,nombre,lat,lon,timestamp,aqi,dominante,NO2,PM25,PM10,O3,SO2,CO,temperatura,humedad,presion,viento,nivel_aqi,fecha,hora
0,madrid,Madrid,40.416775,-3.703790,2026-06-20 08:00:00,53.0,pm25,5.1,53.0,30.0,26.4,1.6,0.1,30.0,33.6,1019.1,2.1,🟡 Moderado,2026-06-20,8
1,barcelona,"Barcelona (Eixample), Catalunya, Spain",41.385343,2.153822,2026-06-20 12:00:00,31.0,o3,8.7,34.0,19.0,30.5,1.1,0.1,30.5,39.5,1023.5,1.0,🟢 Bueno,2026-06-20,12
2,valencia,"Pista de Silla, València, Valencia, Spain",39.456111,-0.375833,2026-06-20 07:00:00,57.0,pm25,3.7,57.0,28.0,19.9,0.6,NaN,26.6,71.0,NaN,0.1,🟡 Moderado,2026-06-20,7
3,sevilla,"Ranilla, Sevilla, Spain",37.384250,-5.959620,2026-06-20 12:00:00,66.0,pm25,1.0,66.0,39.0,29.7,1.6,0.1,33.0,46.0,1013.3,2.5,🟡 Moderado,2026-06-20,12
4,bilbao,"Mazarredo, Bilbao, País Vasco, Spain",43.267500,-2.935200,2026-06-20 11:00:00,0.0,co,3.7,65.0,23.0,NaN,2.6,0.1,28.0,61.0,1019.0,1.0,🟢 Bueno,2026-06-20,11


In [3]:
#============================================
# Celda 3 — Limpieza df_aemet
#============================================
if not AEMET_OK:
    print("⚠️  AEMET vacío — celda saltada (se procesará cuando la API esté disponible)")
else:
    antes = len(df_aemet)
    df_aemet = df_aemet.drop_duplicates(subset=['idema', 'fint'])
    print(f"   Duplicados eliminados: {antes - len(df_aemet)}")

    df_aemet['fint']  = pd.to_datetime(df_aemet['fint'], errors='coerce', utc=True)
    df_aemet['fecha'] = df_aemet['fint'].dt.date
    df_aemet['hora']  = df_aemet['fint'].dt.hour

    cols_num_a = ['lat','lon','alt','ta','tamin','tamax','prec','hr',
                  'vv','vmax','dv','dmax','pres','pres_nmar','vis','inso','ts','tpr']
    for c in cols_num_a:
        if c in df_aemet.columns:
            df_aemet[c] = pd.to_numeric(df_aemet[c], errors='coerce')

    print(f"✅ df_aemet limpio: {df_aemet.shape}")
    print(f"   Nulos: {round(df_aemet.isnull().sum().sum()/df_aemet.size*100,1)}%")
    print(f"   Rango fint: {df_aemet['fint'].min()} → {df_aemet['fint'].max()}")
    df_aemet.head()

⚠️  AEMET vacío — celda saltada (se procesará cuando la API esté disponible)


In [4]:
#============================================
# Celda 4 — Resumen calidad de datos
#============================================
filas = []

if WAQI_OK:
    filas.append({
        'tabla'   : 'waqi_ciudades',
        'filas'   : len(df_waqi),
        'columnas': len(df_waqi.columns),
        'periodos': df_waqi['timestamp'].nunique(),
        'ciudades': df_waqi['ciudad'].nunique(),
        'nulos_%' : round(df_waqi.isnull().sum().sum() / df_waqi.size * 100, 1)
    })

if AEMET_OK:
    col_t = 'fint' if 'fint' in df_aemet.columns else 'fecha'
    filas.append({
        'tabla'   : 'aemet_estaciones',
        'filas'   : len(df_aemet),
        'columnas': len(df_aemet.columns),
        'periodos': df_aemet[col_t].nunique(),
        'ciudades': df_aemet['idema'].nunique(),
        'nulos_%' : round(df_aemet.isnull().sum().sum() / df_aemet.size * 100, 1)
    })

resumen = pd.DataFrame(filas)
print(resumen.to_string(index=False))

        tabla  filas  columnas  periodos  ciudades  nulos_%
waqi_ciudades     30        20         9        30      3.5


In [5]:
#============================================
# Celda 5 — Guardar parquets clean
#============================================
os.makedirs('output/ambiental/02-clean', exist_ok=True)

if WAQI_OK:
    df_waqi.to_parquet('output/ambiental/02-clean/clean_waqi_ciudades.parquet', index=False)
    print(f"✅ Guardado: 02-clean/clean_waqi_ciudades.parquet  ({len(df_waqi)} filas)")

if AEMET_OK:
    df_aemet.to_parquet('output/ambiental/02-clean/clean_aemet_estaciones.parquet', index=False)
    print(f"✅ Guardado: 02-clean/clean_aemet_estaciones.parquet ({len(df_aemet)} filas)")
else:
    print("⚠️  AEMET vacío — clean_aemet_estaciones.parquet no generado")

print("\n🎯 Pipeline 02_ambiental completado")

✅ Guardado: 02-clean/clean_waqi_ciudades.parquet  (30 filas)
⚠️  AEMET vacío — clean_aemet_estaciones.parquet no generado

🎯 Pipeline 02_ambiental completado
